# Telecom Egypt Intelligent Assistant (Self-Contained Groq Version)

This notebook contains the complete source code for the Telecom Egypt RAG pipeline, making it fully self-contained. You can run this directly on **Google Colab** or **Kaggle** without needing to upload the rest of the python scripts from the repository.

It uses the **Groq API** for fast, high-quality generation.

## 1. Install Dependencies

In [1]:
!pip install langchain langchain-community langchain-huggingface langchain-chroma chromadb sentence-transformers transformers edge-tts pypdf python-docx docx2txt pillow beautifulsoup4 requests langchain-groq groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 97.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 125.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 118.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install docx2txt

## 2. Setup API Key and Logging
Get your API key from the [Groq Console](https://console.groq.com/).

In [3]:
import os
import requests
from bs4 import BeautifulSoup
from typing import List
import logging
from PIL import Image
from getpass import getpass

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("Please enter your Groq API Key:")
os.environ["GROQ_API_KEY"] = getpass()

Please enter your Groq API Key:


## 3. Define Prompts

In [4]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

RAG_SYSTEM_PROMPT = """You are a helpful and intelligent assistant for Telecom Egypt (WE).
Your primary task is to answer the user's question based on the provided context and the conversation history.

Follow these STRICT rules:
1. ONLY use the provided context to answer the question. If the answer is not in the context, say "I do not have enough information to answer that based on the provided context." Do not use your own external knowledge.
2. CRITICAL: YOU MUST DETECT THE USER'S LANGUAGE AND REPLY IN THE SAME LANGUAGE. If the user asks in English, you MUST reply in English. If the user asks in Arabic or Egyptian Arabic, you MUST reply in Arabic.
3. DO NOT include source citations inside your response. Provide a natural and continuous answer.
4. REASONING: Before providing your final answer, briefly think step-by-step about how the user's question relates to the provided context. Place your brief reasoning inside <think>...</think> tags. This ensures you do not hallucinate.
5. DOCUMENT LANGUAGE: If the user uploads a document without typing a specific question, you MUST detect the language of the text inside the document. If the document is in Arabic, you MUST reply in Arabic, summarizing or answering based on the document.

Context:
{context}
"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", RAG_SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])


## 4. Document Scrapers and Loaders

In [5]:
import base64
from groq import Groq
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader

def process_image_ocr(file_path: str) -> List[Document]:
    try:
        with open(file_path, "rb") as image_file:
            base64_image = base64.b64encode(image_file.read()).decode('utf-8')

        client = Groq()
        logger.info(f"Running high-quality OCR via Groq Vision for {file_path}...")
        completion = client.chat.completions.create(
            model="llama-3.2-90b-vision-preview",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": "Extract all the text from this image exactly as written. Ensure Arabic and English text is captured perfectly. Output ONLY the extracted text, with absolutely no conversational filler or commentary."
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ],
            temperature=0,
            max_completion_tokens=2048,
        )
        text = completion.choices[0].message.content
        return [Document(page_content=text, metadata={"source": file_path, "type": "image_ocr"})]
    except Exception as e:
        logger.error(f"Error processing image {file_path}: {e}")
        return []

def load_document(file_path: str) -> List[Document]:
    if not os.path.exists(file_path):
        logger.error(f"File not found: {file_path}")
        return []

    ext = os.path.splitext(file_path)[1].lower()
    try:
        if ext == '.pdf': return PyPDFLoader(file_path).load()
        elif ext == '.docx': return Docx2txtLoader(file_path).load()
        elif ext == '.txt': return TextLoader(file_path, encoding='utf-8').load()
        elif ext in ['.png', '.jpg', '.jpeg']: return process_image_ocr(file_path)
        else:
            logger.warning(f"Unsupported file extension: {ext}")
            return []
    except Exception as e:
        logger.error(f"Error loading {file_path}: {e}")
        return []

def scrape_te_page(url: str) -> List[Document]:
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        response.encoding = 'utf-8'

        soup = BeautifulSoup(response.text, 'html.parser')
        for script in soup(["script", "style", "header", "footer", "nav"]):
            script.decompose()

        text = soup.get_text(separator=' ', strip=True)
        logger.info(f"Successfully scraped {len(text)} characters from {url}")
        return [Document(page_content=text, metadata={"source": url, "type": "web_page"})]
    except Exception as e:
        logger.error(f"Error scraping {url}: {e}")
        return []


/tmp/ipykernel_960/2570355966.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader


## 5. Vector Store Configuration

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

def get_embeddings_model() -> HuggingFaceEmbeddings:
    return HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL_NAME,
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True}
    )

def setup_vector_store(documents: List[Document], persist_directory: str = "./data/chroma_db") -> Chroma:
    if not documents:
        return Chroma(collection_name="te_knowledge_base", embedding_function=get_embeddings_model(), persist_directory=persist_directory)

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, length_function=len)
    chunks = text_splitter.split_documents(documents)
    logger.info(f"Split documents into {len(chunks)} chunks.")

    os.makedirs(persist_directory, exist_ok=True)
    return Chroma.from_documents(
        documents=chunks,
        embedding=get_embeddings_model(),
        persist_directory=persist_directory,
        collection_name="te_knowledge_base"
    )

## 6. Run Ingestion (Build Knowledge Base)

In [7]:
# List of high-value TE knowledge targets
faq_urls = [
    "https://www.te.eg/about-te/faq",                  # Main Mobile & USSD FAQs
    "https://te.eg/en/about-te/faq/fixed-broadband",   # Home Internet (WE Space, Routers, Quotas)
    "https://te.eg/en/about-te/faq/fixed-voice"        # Landline (Billing, Installments, Tariffs)
]

all_docs = []
for url in faq_urls:
    logger.info(f"Processing knowledge target: {url}")
    # Using our updated Jina AI scraper from Section 4
    docs = scrape_te_page(url)
    all_docs.extend(docs)

# Set up the Chroma DB and generate embeddings from all scraped FAQ pages
vector_db = setup_vector_store(all_docs, persist_directory="./data/chroma_db")
logger.info(f"Knowledge base successfully populated with {len(all_docs)} source pages!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 7. RAG Pipeline Implementation (Groq)

In [8]:
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(f"[Source: {d.metadata.get('source', 'Unknown')}]\nContent: {d.page_content}" for d in docs)

class GroqRAGPipeline:
    def __init__(self, model_name: str = "llama-3.3-70b-versatile"):
        self.vector_store = Chroma(
            collection_name="te_knowledge_base",
            embedding_function=get_embeddings_model(),
            persist_directory="./data/chroma_db"
        )
        self.retriever = self.vector_store.as_retriever(search_kwargs={"k": 3})
        self.llm = ChatGroq(model_name=model_name, temperature=0.1)

        self.rag_chain = (
            {
                "context": lambda x: format_docs(self.retriever.invoke(x["input"])),
                "input": lambda x: x["input"],
                "chat_history": lambda x: x["chat_history"]
            }
            | qa_prompt
            | self.llm
            | StrOutputParser()
        )

    def query(self, user_input: str, chat_history: list = None) -> dict:
        docs = self.retriever.invoke(user_input)
        answer = self.rag_chain.invoke({"input": user_input, "chat_history": chat_history or []})
        return {"answer": answer, "context": docs}
        
    def stream_query(self, user_input: str, chat_history: list = None):
        docs = self.retriever.invoke(user_input)
        stream = self.rag_chain.stream({"input": user_input, "chat_history": chat_history or []})
        for chunk in stream:
            yield chunk, docs

logger.info("Initializing Groq RAG Pipeline...")
pipeline = GroqRAGPipeline(model_name="llama-3.3-70b-versatile")
logger.info("Pipeline Ready!")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 8. Query the Pipeline

In [9]:
query = "What services does Telecom Egypt offer for personal use?"
result = pipeline.query(query)

print("\nTE Assistant:")
print("-" * 60)
print(result["answer"])
print("-" * 60)

if result["context"]:
    print("\nSources:")
    sources = set(doc.metadata.get('source', 'Unknown') for doc in result["context"])
    for source in sources:
        print(f"  • {source}")


TE Assistant:
------------------------------------------------------------
<think> To answer this question, I need to look at the provided context and identify the services that Telecom Egypt offers for personal use. The context includes information about different types of products and services, including mobile, fixed voice, and fixed broadband. </think>

Telecom Egypt offers several services for personal use, including mobile, fixed voice, and fixed broadband services. These services allow individuals to stay connected and access the internet from their homes or on the go. Additionally, the context mentions prepaid landline subscribers, which suggests that Telecom Egypt also offers landline services for personal use.
------------------------------------------------------------

Sources:
  • https://te.eg/en/about-te/faq/fixed-voice
  • https://te.eg/en/about-te/faq/fixed-broadband
  • https://www.te.eg/about-te/faq


## 9. Speech Integration (ASR & TTS)

In [10]:
import subprocess
import logging
from groq import Groq

logger = logging.getLogger(__name__)

class EgyptianASR:
    def __init__(self):
        self.client = Groq()

    def transcribe(self, audio_path: str) -> str:
        logger.info(f"Transcribing {audio_path} instantly with Groq Whisper...")
        with open(audio_path, "rb") as file:
            transcription = self.client.audio.transcriptions.create(
                file=(audio_path, file.read()),
                model="whisper-large-v3",
                prompt="هذه محادثة لخدمة العملاء باللهجة المصرية.",
                response_format="json",
                language="ar"
            )
        return transcription.text

class HighQualityTTS:
    def __init__(self, voice="ar-EG-SalmaNeural"):
        self.voice = voice

    def synthesize(self, text: str, output_path: str):
        logger.info(f"Synthesizing text using Edge TTS ({self.voice})")
        # Run edge-tts via command line to avoid ALL async/await bugs in Colab!
        subprocess.run(
            ["edge-tts", "--voice", self.voice, "--text", text, "--write-media", output_path],
            check=True
        )


## 10. Test Audio Generation

In [11]:
from IPython.display import Audio

try:
    # Optional: Test ASR (Requires an uploaded audio file like "test_audio.wav")
    # asr = EgyptianASR()
    # transcript = asr.transcribe("test_audio.wav")
    # print(transcript)

    # Initialize TTS and synthesize the result
    tts = HighQualityTTS(voice="ar-EG-SalmaNeural")
    audio_file = "response_output.mp3"

    # Generate audio (using Colab's existing event loop)
    tts.synthesize(result["answer"], audio_file)

    logger.info("Generated audio successfully.")
except Exception as e:
    logger.error(f"Audio processing failed: {e}")

# Audio(audio_file) # Uncomment to play in notebook


## 11. Install Gradio

In [12]:
!pip install gradio


In [13]:
!pip install --upgrade gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 27.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 13.4 MB/s eta 0:00:00
  Attempting uninstall: starlette
    Found existing installation: starlette 0.52.1
    Uninstalling starlette-0.52.1:
      Successfully uninstalled starlette-0.52.1
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    Found existing installation: gradio 5.50.0
    Uninstalling gradio-5.50.0:
      Successfully uninstalled gradio-5.50.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.28.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you hav

## 12. Run the Interactive Frontend in the Notebook

In [14]:
import gradio as gr
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
import re
import uuid
import os

def wrap_rtl(text):
    # Detect if text has Arabic characters
    if re.search("[؀-ۿ]", text):
        return f"<div dir='rtl' style='text-align: right;'>\n\n{text}\n\n</div>"
    return text

def process_interaction(audio_filepath, file_paths, text_input, history):
    uploaded_context = ""
    # 2. Process uploaded files dynamically within the query
    if file_paths:
        from langchain_text_splitters import RecursiveCharacterTextSplitter
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, length_function=len)
        for fp in file_paths:
            # Handle Gradio 4+ file objects safely
            actual_path = fp
            if isinstance(fp, dict) and "name" in fp:
                actual_path = fp["name"]
            elif hasattr(fp, "name"):
                actual_path = getattr(fp, "name")
                
            docs = load_document(actual_path)
            if docs:
                for d in docs:
                    text_content = d.page_content.strip() if d.page_content else ""
                    if text_content:
                        uploaded_context += f"\n[Uploaded Document]: {text_content}\n"
                chunks = text_splitter.split_documents(docs)
                if chunks:
                    pipeline.vector_store.add_documents(chunks)
            else:
                pass # Silent fail if unable to read

    if audio_filepath:
        asr = EgyptianASR()
        user_input = asr.transcribe(audio_filepath)
    else:
        user_input = text_input

    # If the user uploads a document but doesn't type anything, set a default prompt
    if not user_input and uploaded_context:
        user_input = "Please read the uploaded document carefully. If it contains a question, answer it. If not, summarize its contents. Ensure you match the language of the document."

    if not user_input:
        yield "", None, None, history, None
        return

    # We keep a display version of user_input for the UI
    display_user_input = user_input

    # 1. Parse modern Gradio dictionary format into LangChain messages
    chat_history = []
    for msg in history:
        content = msg["content"]
        if isinstance(content, list) or isinstance(content, tuple):
            content = content[0] if len(content) > 0 else ""
        content = str(content)

        if msg["role"] == "user":
            chat_history.append(HumanMessage(content=content))
        elif msg["role"] == "assistant":
            clean_content = content.replace("<div dir='rtl' style='text-align: right;'>\n\n", "").replace("\n\n</div>", "")
            chat_history.append(AIMessage(content=clean_content))

    # Inject the uploaded text into chat history so the LLM sees it directly
    if uploaded_context:
        # Truncate to avoid context window explosion (e.g. max 15000 chars)
        if len(uploaded_context) > 15000:
            uploaded_context = uploaded_context[:15000] + "\n... (truncated due to length)"
        
        chat_history.append(
            SystemMessage(content=f"The user just uploaded a document. Here is its content:\n{uploaded_context}\n\nPlease use this content to answer the user's query.")
        )

    # Add back the display version of user input to history instantly
    history.append({"role": "user", "content": display_user_input})
    history.append({"role": "assistant", "content": ""})
    
    full_answer = ""
    docs = []
    
    # Stream the response
    for chunk, retrieved_docs in pipeline.stream_query(user_input, chat_history=chat_history):
        full_answer += chunk
        docs = retrieved_docs
        
        history[-1]["content"] = full_answer
        yield "", None, None, history, None

    # Detect language of the answer and choose TTS voice
    is_arabic = bool(re.search("[؀-ۿ]", full_answer))
    voice = "ar-EG-SalmaNeural" if is_arabic else "en-US-AriaNeural"
    tts = HighQualityTTS(voice=voice)
    
    # Use a unique filename so Gradio triggers autoplay each time
    out_audio = f"response_{uuid.uuid4().hex}.mp3"

    # Remove the <think> blocks before TTS reading so it doesn't speak its internal reasoning
    spoken_answer = re.sub(r"<think>.*?</think>", "", full_answer, flags=re.DOTALL).strip()

    try:
        if spoken_answer:
            tts.synthesize(spoken_answer, out_audio)
    except Exception as e:
        logger.error(f"TTS Error: {e}")
        out_audio = None

    # Append sources at the bottom
    answer_text = full_answer
    if docs:
        sources = set(d.metadata.get('source', 'Unknown') for d in docs)
        if sources:
            answer_text += "\n\n**Sources:**\n" + "\n".join(f"- {s}" for s in sources)

    # 3. Apply RTL if Arabic
    answer_text_formatted = wrap_rtl(answer_text)
    history[-1]["content"] = answer_text_formatted

    yield "", None, None, history, out_audio

# Set up the Telecom Egypt Purple Theme
we_theme = gr.themes.Soft(
    primary_hue="purple",
    secondary_hue="indigo",
).set(
    button_primary_background_fill="#5b2b82",
    button_primary_background_fill_hover="#4a226b",
    button_primary_text_color="white",
    block_title_text_color="#5b2b82"
)

with gr.Blocks(theme=we_theme, css=".gradio-container {max-width: 900px; margin: auto;}") as demo:
    gr.Markdown(
        "<h1 style='text-align: center; color: #5b2b82;'>Telecom Egypt Intelligent Assistant</h1>"
    )
    chatbot = gr.Chatbot(label="TE Assistant", height=500)

    # 5. Make audio_output visible so the user can play it manually if autoplay is blocked
    audio_output = gr.Audio(
        label="Assistant Voice Response", autoplay=False, visible=True
    )

    with gr.Row():
        with gr.Column(scale=8):
            txt = gr.Textbox(
                show_label=False,
                placeholder="Type your message here...",
                container=False
            )
        with gr.Column(scale=1, min_width=80):
            # 1. Send button explicitly
            submit_btn = gr.Button("Send", variant="primary")

    with gr.Row():
        audio_in = gr.Audio(
            sources=["microphone"], type="filepath", label="Record Voice (Optional)"
        )
        file_in = gr.File(label="Attach Documents (Optional)", file_count="multiple")

    # Disable buttons during processing, then enable when done
    submit_btn.click(
        fn=lambda: gr.update(interactive=False),
        inputs=None,
        outputs=[submit_btn]
    ).then(
        fn=process_interaction,
        inputs=[audio_in, file_in, txt, chatbot],
        outputs=[txt, audio_in, file_in, chatbot, audio_output],
    ).then(
        fn=lambda: gr.update(interactive=True),
        inputs=None,
        outputs=[submit_btn]
    )
    
    txt.submit(
        fn=lambda: gr.update(interactive=False),
        inputs=None,
        outputs=[submit_btn]
    ).then(
        fn=process_interaction,
        inputs=[audio_in, file_in, txt, chatbot],
        outputs=[txt, audio_in, file_in, chatbot, audio_output],
    ).then(
        fn=lambda: gr.update(interactive=True),
        inputs=None,
        outputs=[submit_btn]
    )

# Launch cleanly
demo.launch(share=True, debug=True)


/tmp/ipykernel_960/1228671348.py:137: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=we_theme, css=".gradio-container {max-width: 900px; margin: auto;}") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://575ee95dda977f6def.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://575ee95dda977f6def.gradio.live
